# Задание 2. Проницаемость водорода

Рассмотрим перенос водорода через пластину толщиной $L=0.3$ мм при температуре 500 К.

На входной поверхности концентрация задаётся законом Сивертса:

$$
c=S\sqrt{P}.
$$

На выходной поверхности концентрация равна нулю. Рассчитанный выходной поток сравнивается с аналитическим решением.


In [ ]:
import festim as F
import numpy as np
import matplotlib.pyplot as plt

model = F.HydrogenTransportProblem()

H = F.Species("H")
model.species = [H]

L = 3e-4
model.mesh = F.Mesh1D(
    vertices=np.linspace(0.0, L, num=1001),
)

material = F.Material(
    D_0=1.9e-7,
    E_D=0.2,
)

volume = F.VolumeSubdomain1D(
    id=1,
    borders=[0.0, L],
    material=material,
)
left = F.SurfaceSubdomain1D(id=1, x=0.0)
right = F.SurfaceSubdomain1D(id=2, x=L)

model.subdomains = [volume, left, right]
model.temperature = 500.0

## Граничные условия


In [ ]:
P_up = 100.0

model.boundary_conditions = [
    F.SievertsBC(
        subdomain=left,
        S_0=4.02e21,
        E_S=1.04,
        pressure=P_up,
        species=H,
    ),
    F.FixedConcentrationBC(
        subdomain=right,
        value=0.0,
        species=H,
    ),
]

## Расчёт выходного потока


In [ ]:
model.settings = F.Settings(
    atol=1e-2,
    rtol=1e-10,
    final_time=100.0,
    stepsize=F.Stepsize(1 / 20),
)

permeation_flux = F.SurfaceFlux(
    field=H,
    surface=right,
)
model.exports = [permeation_flux]

model.initialise()
model.run()

## Сравнение с аналитическим решением

$$
J(t)=\frac{\sqrt{P_\mathrm{up}}\Phi}{L}
\left[
1+2\sum_{n=1}^{\infty}
(-1)^n
\exp\left(-\frac{\pi^2Dn^2t}{L^2}\right)
\right],
$$

где $\Phi=DS$ — проницаемость.


In [ ]:
def analytical_flux(t, pressure, permeability, thickness, diffusivity):
    n = np.arange(1, 10000)[:, np.newaxis]
    series = np.sum(
        (-1) ** n
        * np.exp(
            -(np.pi * n) ** 2
            * diffusivity
            * t
            / thickness**2
        ),
        axis=0,
    )
    return (
        np.sqrt(pressure)
        * permeability
        / thickness
        * (1.0 + 2.0 * series)
    )


times = np.asarray(permeation_flux.t)

D = 1.9e-7 * np.exp(-0.2 / (F.k_B * 500.0))
S = 4.02e21 * np.exp(-1.04 / (F.k_B * 500.0))

plt.scatter(
    times,
    np.abs(permeation_flux.data),
    alpha=0.25,
    label="FESTIM",
)
plt.plot(
    times,
    analytical_flux(
        times,
        pressure=P_up,
        permeability=D * S,
        thickness=L,
        diffusivity=D,
    ),
    label="Аналитическое решение",
    color="tab:red",
    lw=2,
)

plt.ylim(bottom=0)
plt.xlabel("Время, с")
plt.ylabel(r"Выходной поток H, м$^{-2}$ с$^{-1}$")
plt.legend()
plt.show()